# Torchvision — Deep Learning for Images with PyTorch

## What Is This Notebook About?

In the OpenCV notebook we learned to *manipulate* images (resize, blur, detect edges). Now we want computers to *understand* images — "Is this a cat or a dog?" "Which pixels belong to a person?"

**Torchvision** is PyTorch's official computer vision library. It gives you:
- **Pre-trained models** (ResNet, VGG, EfficientNet — trained on 1 million+ images)
- **Datasets** (ImageNet, CIFAR-10, MNIST — ready to download)
- **Transforms** (data augmentation pipeline — rotate, crop, normalize)

---

## Why Should You Care?

| Task | Torchvision Model |
|---|---|
| Photo classification (cat/dog/car) | ResNet, EfficientNet |
| Object detection | Faster R-CNN, RetinaNet |
| Segmentation | DeepLab, FCN |
| Medical imaging diagnosis | Transfer learned ResNet |
| Satellite imagery analysis | Custom CNN |
| Product defect detection | Fine-tuned EfficientNet |

---

## Prerequisites

- Python basics, NumPy
- The PyTorch notebook (tensors, nn.Module, training loop)
- OpenCV notebook (what images are)

---

## Table of Contents

1. [Setup](#1-setup)
2. [Transforms — Data Augmentation Pipeline](#2-transforms)
3. [Datasets — Loading Image Data](#3-datasets)
4. [Pre-trained Models Zoo](#4-models)
5. [Transfer Learning — Fine-Tuning ResNet](#5-transfer)
6. [Training a CNN from Scratch on CIFAR-10](#6-cnn-scratch)
7. [Grad-CAM — What Did the Model See?](#7-gradcam)
8. [Mini Project — Plant Disease Classifier](#8-mini-project)
9. [Common Pitfalls](#9-pitfalls)
10. [Interview Q&A](#10-interview)
11. [Resources](#11-resources)
12. [Summary](#12-summary)

---
## 1. Setup <a id='1-setup'></a>

In [ ]:
# pip install torch torchvision matplotlib numpy scikit-learn

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split

import torchvision
import torchvision.transforms as transforms
import torchvision.transforms.v2 as v2      # New API (PyTorch ≥ 2.0)
from torchvision import datasets, models
from torchvision.utils import make_grid

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Device: use GPU if available, else CPU
device = torch.device('cuda' if torch.cuda.is_available() else
                      'mps'  if torch.backends.mps.is_available() else 'cpu')

print(f"PyTorch version:    {torch.__version__}")
print(f"Torchvision version:{torchvision.__version__}")
print(f"Device:             {device}")

np.random.seed(42)
torch.manual_seed(42)

---
## 2. Transforms — Data Augmentation Pipeline <a id='2-transforms'></a>

### The Photography Studio Analogy

Imagine training a model that must recognize dogs. If all training photos show dogs sitting still, facing forward, in daylight — the model fails on a dog running sideways at night.

**Transforms = artificial variations** to make models robust:
- Randomly flip, crop, rotate, change brightness → model learns the *concept* of dog, not specific angles
- Normalize → converts pixel values to have mean≈0, std≈1 → faster, more stable training

**Rule: heavy augmentation for training, only normalize for validation.**

In [ ]:
# ── Build transform pipelines ─────────────────────────────────────────────────

# ImageNet normalization constants (precomputed from 1.2M images)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Training transform: augmentation + normalize
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),        # Random crop → resize to 224×224
    transforms.RandomHorizontalFlip(p=0.5),   # 50% chance flip
    transforms.RandomRotation(degrees=15),    # Rotate up to ±15°
    transforms.ColorJitter(
        brightness=0.3, contrast=0.3,          # Vary brightness/contrast
        saturation=0.2, hue=0.1               # Vary color
    ),
    transforms.ToTensor(),                    # PIL Image → FloatTensor [0,1], shape (C,H,W)
    transforms.Normalize(IMAGENET_MEAN,       # Subtract mean, divide by std
                         IMAGENET_STD),
])

# Validation/test transform: NO augmentation, just resize + normalize
val_transform = transforms.Compose([
    transforms.Resize(256),                   # Resize shorter edge to 256
    transforms.CenterCrop(224),               # Take 224×224 center
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# CIFAR-10 transforms (smaller images: 32×32)
cifar_train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),     # Pad 4 px, random 32×32 crop
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.4914, 0.4822, 0.4465],
                         [0.2023, 0.1994, 0.2010]),  # CIFAR-10 stats
])

cifar_val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.4914, 0.4822, 0.4465],
                         [0.2023, 0.1994, 0.2010]),
])

print("Training transform pipeline:")
for t in train_transform.transforms:
    print(f"  → {t.__class__.__name__}")
print("\nValidation transform pipeline:")
for t in val_transform.transforms:
    print(f"  → {t.__class__.__name__}")

In [ ]:
# ── Visualize transforms on a synthetic image ─────────────────────────────────

from PIL import Image

# Create a synthetic PIL image (colored shapes) to show transform effects
import io
import cv2

synth_bgr = np.zeros((256, 256, 3), dtype=np.uint8)
cv2.circle(synth_bgr, (80, 80), 50, (30, 120, 200), -1)
cv2.rectangle(synth_bgr, (140, 40), (220, 120), (200, 80, 30), -1)
cv2.ellipse(synth_bgr, (128, 190), (90, 50), 0, 0, 360, (50, 180, 50), -1)
cv2.putText(synth_bgr, 'Sample', (60, 240), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (220,220,220), 2)

synth_rgb = cv2.cvtColor(synth_bgr, cv2.COLOR_BGR2RGB)
pil_img = Image.fromarray(synth_rgb)

# Apply each transform individually to show effect
individual_transforms = [
    ('Original', transforms.Compose([transforms.ToTensor()])),
    ('RandomHFlip', transforms.Compose([transforms.RandomHorizontalFlip(p=1.0), transforms.ToTensor()])),
    ('RandomRotation 30°', transforms.Compose([transforms.RandomRotation(30), transforms.ToTensor()])),
    ('ColorJitter', transforms.Compose([transforms.ColorJitter(0.5, 0.5, 0.4, 0.2), transforms.ToTensor()])),
    ('RandomCrop+Pad', transforms.Compose([transforms.Pad(20), transforms.RandomCrop(256), transforms.ToTensor()])),
    ('RandomResizedCrop', transforms.Compose([transforms.RandomResizedCrop(224, scale=(0.5, 0.8)), transforms.ToTensor()])),
    ('Grayscale', transforms.Compose([transforms.Grayscale(3), transforms.ToTensor()])),
    ('GaussianBlur', transforms.Compose([transforms.GaussianBlur(11, sigma=3), transforms.ToTensor()])),
]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, (name, tfm) in zip(axes.flatten(), individual_transforms):
    tensor = tfm(pil_img)
    img_np = tensor.permute(1, 2, 0).numpy()
    img_np = np.clip(img_np, 0, 1)
    ax.imshow(img_np)
    ax.set_title(name, fontsize=9)
    ax.axis('off')

plt.suptitle('Torchvision Transforms Visualized\n'
             '(Each run produces different result — stochastic transforms)', fontsize=12)
plt.tight_layout()
plt.show()

print("KEY: ToTensor() does TWO things:")
print("  1. Rearranges axes: (H, W, C) → (C, H, W)  ← PyTorch expects channels first")
print("  2. Scales values:   [0, 255] → [0.0, 1.0]")
print()
print("After Normalize([mean], [std]):")
print("  pixel = (pixel - mean) / std")
print("  This centers pixel values around 0, stabilizing training")

---
## 3. Datasets — Loading Image Data <a id='3-datasets'></a>

### The Filing Cabinet Analogy

A `Dataset` is like a filing cabinet: each drawer (index) contains one (image, label) pair. A `DataLoader` is like an office worker who opens multiple drawers simultaneously, shuffles the files, and hands them to you in batches.

In [ ]:
# ── Built-in datasets ─────────────────────────────────────────────────────────

# CIFAR-10: 60,000 color images (32×32), 10 classes
# Downloads ~170MB to ./data/ on first run
print("Loading CIFAR-10...")
try:
    cifar_train = datasets.CIFAR10(
        root='./data',
        train=True,
        download=True,
        transform=cifar_train_transform
    )
    cifar_test = datasets.CIFAR10(
        root='./data',
        train=False,
        download=True,
        transform=cifar_val_transform
    )

    CIFAR_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                     'dog', 'frog', 'horse', 'ship', 'truck']

    print(f"Train set: {len(cifar_train):,} images")
    print(f"Test set:  {len(cifar_test):,} images")
    print(f"Classes:   {CIFAR_CLASSES}")

    img, label = cifar_train[0]
    print(f"\nSingle sample:")
    print(f"  Image tensor shape: {img.shape}   (C=3, H=32, W=32)")
    print(f"  Label:              {label} ({CIFAR_CLASSES[label]})")
    print(f"  Pixel value range:  [{img.min():.2f}, {img.max():.2f}]  (normalized)")
    CIFAR_AVAILABLE = True

except Exception as e:
    print(f"Could not download CIFAR-10: {e}")
    print("Using synthetic data instead.")
    CIFAR_AVAILABLE = False

In [ ]:
# ── Custom Dataset class for your own image folders ───────────────────────────

class SyntheticImageDataset(Dataset):
    """
    Synthetic dataset mimicking a real image folder structure.
    In real projects, use datasets.ImageFolder for this pattern:

    data/
    ├── train/
    │   ├── cats/  ← folder name = class name
    │   │   ├── img001.jpg
    │   │   └── img002.jpg
    │   └── dogs/
    │       └── img003.jpg
    └── val/
        ├── cats/
        └── dogs/

    Usage:
        dataset = datasets.ImageFolder('data/train', transform=train_transform)
    """

    def __init__(self, n_samples=1000, n_classes=10, img_size=32, transform=None):
        self.n_samples  = n_samples
        self.n_classes  = n_classes
        self.img_size   = img_size
        self.transform  = transform

        # Generate: each class has a characteristic mean color
        self.class_colors = [
            np.array([200, 80, 80]),   # class 0: reddish
            np.array([80, 200, 80]),   # class 1: greenish
            np.array([80, 80, 200]),   # class 2: blueish
            np.array([200, 200, 80]),  # class 3: yellowish
            np.array([200, 80, 200]),  # class 4: pinkish
            np.array([80, 200, 200]),  # class 5: cyanish
            np.array([160, 120, 80]),  # class 6: brownish
            np.array([120, 80, 160]),  # class 7: purple
            np.array([80, 160, 120]),  # class 8: teal
            np.array([160, 160, 160]), # class 9: gray
        ][:n_classes]

        self.labels = np.random.randint(0, n_classes, n_samples)

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        label = self.labels[idx]
        # Create image with class-specific color + noise
        base_color = self.class_colors[label]
        noise = np.random.randint(-40, 40, (self.img_size, self.img_size, 3))
        pixel_values = np.clip(base_color + noise, 0, 255).astype(np.uint8)

        img = Image.fromarray(pixel_values)

        if self.transform:
            img = self.transform(img)
        else:
            img = transforms.ToTensor()(img)

        return img, label


# Create synthetic dataset
synth_dataset = SyntheticImageDataset(n_samples=2000, n_classes=10)
print(f"Synthetic dataset: {len(synth_dataset)} images, {synth_dataset.n_classes} classes")

# DataLoader
loader = DataLoader(
    synth_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,      # 0=single process (safe in notebooks; use 4+ in scripts)
    pin_memory=(device.type == 'cuda'),  # Faster GPU transfer
    drop_last=False     # Keep the last incomplete batch
)

# Inspect one batch
imgs, labels = next(iter(loader))
print(f"\nBatch shape: {imgs.shape}  (batch_size=32, C=3, H=32, W=32)")
print(f"Labels:      {labels[:8].tolist()}")
print(f"Pixel range: [{imgs.min():.3f}, {imgs.max():.3f}]")

In [ ]:
# ── Visualize a batch from the DataLoader ─────────────────────────────────────

def show_batch(images, labels, class_names=None, n_show=16, figsize=(14, 6)):
    """Display first n_show images from a batch."""
    images = images[:n_show]
    labels = labels[:n_show]

    # Unnormalize if needed (assuming CIFAR-10 normalization)
    mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
    std  = torch.tensor([0.2023, 0.1994, 0.2010]).view(3, 1, 1)

    # Check if images look normalized (values outside [0,1])
    if images.min() < -0.1:
        images = images * std + mean  # unnormalize

    images = torch.clamp(images, 0, 1)

    n_cols = min(8, n_show)
    n_rows = (n_show + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    if n_rows == 1:
        axes = [axes]

    for i in range(n_show):
        r, c = i // n_cols, i % n_cols
        ax = axes[r][c] if n_rows > 1 else axes[0][c]
        img_np = images[i].permute(1, 2, 0).numpy()
        ax.imshow(img_np)
        lbl = int(labels[i])
        name = class_names[lbl] if class_names else str(lbl)
        ax.set_title(name, fontsize=8)
        ax.axis('off')

    # Hide unused subplots
    total_axes = n_rows * n_cols
    for i in range(n_show, total_axes):
        r, c = i // n_cols, i % n_cols
        ax = axes[r][c] if n_rows > 1 else axes[0][c]
        ax.axis('off')

    plt.suptitle('Batch from DataLoader', fontsize=12)
    plt.tight_layout()
    plt.show()


if CIFAR_AVAILABLE:
    cifar_loader = DataLoader(cifar_train, batch_size=16, shuffle=True)
    imgs, labels = next(iter(cifar_loader))
    show_batch(imgs, labels, CIFAR_CLASSES, n_show=16, figsize=(14, 4))
else:
    imgs, labels = next(iter(loader))
    show_batch(imgs, labels, n_show=16, figsize=(14, 4))

---
## 4. Pre-trained Models Zoo <a id='4-models'></a>

### The Expert Consultant Analogy

Training ResNet-50 on ImageNet takes **hundreds of GPU hours** and millions of images. Pre-trained models are like hiring an expert who already learned the fundamental visual concepts (edges, textures, shapes, objects) over years of training. You just **adapt** their knowledge to your specific task.

**ImageNet**: 1.28M images, 1000 classes — the benchmark dataset for visual recognition.

In [ ]:
# ── Explore torchvision model zoo ─────────────────────────────────────────────

model_comparison = [
    ('ResNet-18',      'resnet18',       11.7,   1.8,  'Fast, lightweight'),
    ('ResNet-50',      'resnet50',       25.6,   4.1,  'Great balance'),
    ('ResNet-101',     'resnet101',      44.5,   7.8,  'High accuracy'),
    ('VGG-16',         'vgg16',          138.0,  15.5, 'Old but proven'),
    ('EfficientNet-B0','efficientnet_b0', 5.3,   0.4,  'Small & accurate'),
    ('EfficientNet-B4','efficientnet_b4', 19.3,  1.5,  'Best efficiency'),
    ('MobileNetV3-L',  'mobilenet_v3_large', 5.5, 0.22,'Mobile deployment'),
    ('ViT-B/16',       'vit_b_16',       86.6,  17.6, 'Vision Transformer'),
]

print(f"{'Model':20s} {'Params(M)':10s} {'GFLOPs':8s} {'Notes'}")
print("─" * 65)
for name, _, params, gflops, notes in model_comparison:
    print(f"{name:20s} {params:10.1f} {gflops:8.1f} {notes}")

print("\n")
print("Practical choice guide:")
print("  Mobile/edge deployment:   MobileNetV3, EfficientNet-B0")
print("  Fast experimentation:     ResNet-18, ResNet-50")
print("  Maximum accuracy:         EfficientNet-B4, ViT-B/16")
print("  Classic baseline:         ResNet-50 (the 'standard' choice)")

# Load a pre-trained ResNet-18
print("\nLoading pre-trained ResNet-18...")
resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
resnet.eval()

# Count parameters
total_params = sum(p.numel() for p in resnet.parameters())
trainable_params = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
# ── Run inference with pre-trained model ─────────────────────────────────────

# Load ImageNet class labels
IMAGENET_CLASSES = models.ResNet18_Weights.IMAGENET1K_V1.meta['categories']

# Create synthetic test image (a reddish blob → might predict something red)
synth_bgr = np.zeros((224, 224, 3), dtype=np.uint8)
import cv2
cv2.circle(synth_bgr, (112, 112), 80, (30, 50, 220), -1)   # Red circle
cv2.rectangle(synth_bgr, (50, 160), (170, 210), (80, 180, 80), -1)  # green rect
synth_rgb = cv2.cvtColor(synth_bgr, cv2.COLOR_BGR2RGB)
pil_test = Image.fromarray(synth_rgb)

# Preprocess
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

input_tensor = preprocess(pil_test).unsqueeze(0)   # Add batch dimension: (1, 3, 224, 224)

# Forward pass (no gradient computation needed for inference)
with torch.no_grad():
    logits = resnet(input_tensor)   # shape: (1, 1000)

# Convert to probabilities
probs = F.softmax(logits, dim=1)[0]   # shape: (1000,)
top5_probs, top5_idx = torch.topk(probs, 5)

print("ResNet-18 Top-5 Predictions on Synthetic Image:")
print(f"  (Note: synthetic shapes won't match real ImageNet classes well)")
print()
for prob, idx in zip(top5_probs, top5_idx):
    class_name = IMAGENET_CLASSES[idx]
    bar = '█' * int(prob.item() * 50)
    print(f"  [{prob.item()*100:5.2f}%] {class_name:30s} {bar}")

print(f"\nOutput tensor shape: {logits.shape}  (1 image × 1000 ImageNet classes)")
print(f"Sum of probabilities: {probs.sum().item():.4f}  (should be ≈ 1.0)")

---
## 5. Transfer Learning — Fine-Tuning ResNet <a id='5-transfer'></a>

### The Expert Retraining Analogy

ResNet-18 knows 1000 ImageNet categories. You want to classify 10 plant diseases. Strategy:

1. **Freeze** the body (existing feature extraction layers) — don't change what it already knows
2. **Replace** the head (final classification layer) — 1000 → 10 outputs
3. **Train** only the new head first — fast, no overfitting
4. **Unfreeze** last few layers + retrain at low LR — fine-tune to your domain

This works because early CNN layers learn universal features (edges, corners, textures) that apply to ANY image domain.

In [ ]:
# ── Transfer learning setup ───────────────────────────────────────────────────

N_CLASSES = 10  # Our task: 10 plant disease classes

def build_transfer_model(n_classes, freeze_backbone=True):
    """
    Load ResNet-18, replace final layer for new task.
    
    Args:
        n_classes: number of output classes
        freeze_backbone: if True, only train the new classifier head
    """
    # 1. Load pre-trained model
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    # 2. Freeze backbone (all layers won't update during training)
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    # 3. Replace the final fully-connected layer
    # Original: Linear(512, 1000)  →  New: Linear(512, n_classes)
    in_features = model.fc.in_features   # 512 for ResNet-18
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(256, n_classes)
    )
    # The new layers always have requires_grad=True by default

    return model


model = build_transfer_model(N_CLASSES, freeze_backbone=True)
model = model.to(device)

# Count trainable vs frozen parameters
total  = sum(p.numel() for p in model.parameters())
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
trainable = total - frozen

print(f"Transfer Learning Setup:")
print(f"  Total parameters:     {total:>12,}")
print(f"  Frozen (backbone):    {frozen:>12,}  ({frozen/total*100:.1f}%)")
print(f"  Trainable (head):     {trainable:>12,}  ({trainable/total*100:.1f}%)")
print(f"\nThis is the power of transfer learning:")
print(f"  Train only {trainable:,} params instead of {total:,} → much faster!")

# Verify output shape
dummy = torch.randn(4, 3, 224, 224).to(device)
with torch.no_grad():
    out = model(dummy)
print(f"\nOutput shape for batch of 4: {out.shape}  (4 images × {N_CLASSES} classes)")

In [ ]:
# ── Training loop for transfer learning ──────────────────────────────────────

# Use synthetic data (replace with your real ImageFolder dataset in practice)
synth_train = SyntheticImageDataset(n_samples=2000, n_classes=N_CLASSES)
synth_val   = SyntheticImageDataset(n_samples=400,  n_classes=N_CLASSES)

train_loader = DataLoader(synth_train, batch_size=64, shuffle=True,  num_workers=0)
val_loader   = DataLoader(synth_val,   batch_size=64, shuffle=False, num_workers=0)

# Optimizer: only pass trainable parameters!
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)
criterion = nn.CrossEntropyLoss()
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)


def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = total_correct = total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss    += loss.item() * imgs.size(0)
        total_correct += (logits.argmax(1) == labels).sum().item()
        total         += imgs.size(0)
    return total_loss / total, total_correct / total


def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = total_correct = total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs)
            loss = criterion(logits, labels)
            total_loss    += loss.item() * imgs.size(0)
            total_correct += (logits.argmax(1) == labels).sum().item()
            total         += imgs.size(0)
    return total_loss / total, total_correct / total


# Phase 1: Train only the new head (5 epochs)
EPOCHS_PHASE1 = 5
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

print("Phase 1: Training head only (backbone frozen)")
print(f"{'Epoch':>6} {'Train Loss':>11} {'Train Acc':>10} {'Val Loss':>10} {'Val Acc':>9} {'LR':>10}")
print("─" * 65)

for epoch in range(1, EPOCHS_PHASE1 + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    va_loss, va_acc = eval_epoch(model, val_loader, criterion, device)
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(va_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(va_acc)

    print(f"{epoch:>6} {tr_loss:>11.4f} {tr_acc:>10.4f} {va_loss:>10.4f} {va_acc:>9.4f} {current_lr:>10.6f}")

print("\nPhase 1 complete!")

In [ ]:
# ── Phase 2: Unfreeze and fine-tune ──────────────────────────────────────────

# Unfreeze the last ResNet layer block (layer4)
for name, param in model.named_parameters():
    if 'layer4' in name or 'fc' in name:
        param.requires_grad = True

trainable_phase2 = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Phase 2: Unfreezing layer4 + fc")
print(f"  Trainable parameters: {trainable_phase2:,} (was {trainable:,})")

# Use much lower learning rate to not destroy pretrained weights
optimizer_phase2 = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4   # 10× smaller than phase 1
)
scheduler_phase2 = optim.lr_scheduler.CosineAnnealingLR(optimizer_phase2, T_max=5)

EPOCHS_PHASE2 = 5
print(f"\n{'Epoch':>6} {'Train Loss':>11} {'Train Acc':>10} {'Val Loss':>10} {'Val Acc':>9}")
print("─" * 55)

for epoch in range(1, EPOCHS_PHASE2 + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer_phase2, criterion, device)
    va_loss, va_acc = eval_epoch(model, val_loader, criterion, device)
    scheduler_phase2.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(va_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(va_acc)

    print(f"{epoch+EPOCHS_PHASE1:>6} {tr_loss:>11.4f} {tr_acc:>10.4f} {va_loss:>10.4f} {va_acc:>9.4f}")

# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
epochs = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs, history['train_loss'], 'b-o', ms=4, label='Train')
axes[0].plot(epochs, history['val_loss'],   'r-o', ms=4, label='Val')
axes[0].axvline(EPOCHS_PHASE1 + 0.5, color='gray', ls='--', label='Phase 1→2')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curve'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, [a*100 for a in history['train_acc']], 'b-o', ms=4, label='Train')
axes[1].plot(epochs, [a*100 for a in history['val_acc']],   'r-o', ms=4, label='Val')
axes[1].axvline(EPOCHS_PHASE1 + 0.5, color='gray', ls='--', label='Phase 1→2')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy Curve'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Transfer Learning: Phase 1 (head only) → Phase 2 (backbone fine-tune)', fontsize=12)
plt.tight_layout()
plt.show()

---
## 6. Training a CNN from Scratch on CIFAR-10 <a id='6-cnn-scratch'></a>

### When to Train from Scratch

Use transfer learning whenever possible. Train from scratch only when:
- Your images are fundamentally different from ImageNet (e.g., X-rays, satellite imagery, microscopy)
- You have a very large custom dataset (100K+ images)
- The pre-trained domain knowledge would actually hurt (e.g., texture-less scientific images)

In [ ]:
# ── Custom CNN architecture for CIFAR-10 (32×32 images) ──────────────────────

class CIFARNet(nn.Module):
    """
    Custom CNN for CIFAR-10 (32×32 input images).
    
    Architecture:
        Conv Block 1: Conv→BN→ReLU→MaxPool
        Conv Block 2: Conv→BN→ReLU→MaxPool  
        Conv Block 3: Conv→BN→ReLU (no pool — image too small)
        Classifier:   FC→ReLU→Dropout→FC
    """

    def __init__(self, n_classes=10):
        super().__init__()

        # Feature extraction
        self.features = nn.Sequential(
            # Block 1: 32×32×3 → 16×16×64
            nn.Conv2d(3, 64, kernel_size=3, padding=1),  # same padding
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),   # 32×32 → 16×16
            nn.Dropout2d(0.25),

            # Block 2: 16×16×64 → 8×8×128
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),   # 16×16 → 8×8
            nn.Dropout2d(0.25),

            # Block 3: 8×8×128 → 8×8×256 (no pool)
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )

        # Global Average Pooling: 8×8×256 → 256
        # Better than Flatten + FC for small images
        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(256, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, n_classes)
        )

    def forward(self, x):
        x = self.features(x)           # (batch, 256, 8, 8)
        x = self.gap(x)                # (batch, 256, 1, 1)
        x = x.flatten(1)               # (batch, 256)
        return self.classifier(x)      # (batch, n_classes)


cnn = CIFARNet(n_classes=10).to(device)

# Print architecture summary
total = sum(p.numel() for p in cnn.parameters())
print(f"CIFARNet: {total:,} trainable parameters")
print(f"(ResNet-18 has 11.7M — this is much smaller for 32×32 images)")

# Verify output shape
dummy = torch.randn(4, 3, 32, 32).to(device)
with torch.no_grad():
    out = cnn(dummy)
print(f"\nOutput for batch of 4: {out.shape}")

In [ ]:
# ── Quick training demo on synthetic data ─────────────────────────────────────
# (Replace with cifar_train/cifar_test for real CIFAR-10 training)

synth_cnn_train = SyntheticImageDataset(n_samples=3000, n_classes=10, img_size=32)
synth_cnn_val   = SyntheticImageDataset(n_samples=600,  n_classes=10, img_size=32)

cnn_train_loader = DataLoader(synth_cnn_train, batch_size=64, shuffle=True,  num_workers=0)
cnn_val_loader   = DataLoader(synth_cnn_val,   batch_size=64, shuffle=False, num_workers=0)

optimizer_cnn = optim.AdamW(cnn.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler_cnn = optim.lr_scheduler.OneCycleLR(
    optimizer_cnn,
    max_lr=3e-3,
    steps_per_epoch=len(cnn_train_loader),
    epochs=10
)

cnn_history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

print("Training CIFARNet from scratch on synthetic data")
print(f"{'Epoch':>6} {'Train Loss':>11} {'Train Acc':>10} {'Val Loss':>10} {'Val Acc':>9}")
print("─" * 55)

for epoch in range(1, 11):
    # Training
    cnn.train()
    tr_loss = tr_correct = tr_total = 0
    for imgs, labels in cnn_train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer_cnn.zero_grad()
        logits = cnn(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(cnn.parameters(), max_norm=1.0)  # gradient clipping
        optimizer_cnn.step()
        scheduler_cnn.step()
        tr_loss    += loss.item() * imgs.size(0)
        tr_correct += (logits.argmax(1) == labels).sum().item()
        tr_total   += imgs.size(0)

    va_loss, va_acc = eval_epoch(cnn, cnn_val_loader, criterion, device)

    cnn_history['train_loss'].append(tr_loss / tr_total)
    cnn_history['train_acc'].append(tr_correct / tr_total)
    cnn_history['val_loss'].append(va_loss)
    cnn_history['val_acc'].append(va_acc)

    print(f"{epoch:>6} {tr_loss/tr_total:>11.4f} {tr_correct/tr_total:>10.4f} {va_loss:>10.4f} {va_acc:>9.4f}")

print("\nDone! On real CIFAR-10 (50K images), this architecture achieves ~88% test accuracy.")

---
## 7. Grad-CAM — What Did the Model See? <a id='7-gradcam'></a>

### The Highlight Pen Analogy

When a doctor diagnoses a chest X-ray, you can ask "Which part of the X-ray made you say this?" **Grad-CAM** does the same for CNNs: it highlights the image regions that most influenced the model's prediction.

**Algorithm:**
1. Run forward pass, record activations of last conv layer
2. Run backward pass from target class score
3. Global average pool gradients → class-specific weights
4. Weighted sum of activations → heatmap

In [ ]:
# ── Grad-CAM implementation ───────────────────────────────────────────────────

import cv2 as cv2_local

class GradCAM:
    """
    Gradient-weighted Class Activation Mapping.
    Generates heatmaps showing what image regions influenced the prediction.
    
    Paper: https://arxiv.org/abs/1610.02391
    """

    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)

    def generate(self, input_tensor, target_class=None):
        """
        Generate Grad-CAM heatmap for input image.
        
        Args:
            input_tensor: (1, C, H, W) preprocessed image tensor
            target_class: class index to explain (None = use predicted class)
        """
        self.model.eval()

        # Forward pass
        logits = self.model(input_tensor)

        if target_class is None:
            target_class = logits.argmax(1).item()

        # Backward pass for target class
        self.model.zero_grad()
        logits[0, target_class].backward()

        # Pool gradients over spatial dimensions → (C,)
        weights = self.gradients.mean(dim=[2, 3])   # global average pool

        # Weighted combination of activation maps
        cam = (weights[0, :, None, None] * self.activations[0]).sum(dim=0)

        # ReLU: only positive influence
        cam = F.relu(cam)

        # Normalize to [0, 1]
        cam -= cam.min()
        if cam.max() > 0:
            cam /= cam.max()

        return cam.cpu().numpy(), target_class, float(F.softmax(logits, dim=1)[0, target_class])


def overlay_cam(img_np, cam, alpha=0.5):
    """Overlay Grad-CAM heatmap on original image."""
    # Resize CAM to image size
    h, w = img_np.shape[:2]
    cam_resized = cv2_local.resize(cam, (w, h))

    # Convert to heatmap
    heatmap = cv2_local.applyColorMap(
        (cam_resized * 255).astype(np.uint8),
        cv2_local.COLORMAP_JET
    )
    heatmap_rgb = cv2_local.cvtColor(heatmap, cv2_local.COLOR_BGR2RGB)

    # Blend
    overlay = (alpha * heatmap_rgb + (1 - alpha) * img_np * 255).astype(np.uint8)
    return overlay, heatmap_rgb


# ── Apply Grad-CAM to our CIFARNet ───────────────────────────────────────────

# Target: last conv layer (last element of features block before GAP)
# Find the last Conv2d in features
last_conv = None
for module in cnn.features.modules():
    if isinstance(module, nn.Conv2d):
        last_conv = module

gradcam = GradCAM(cnn, last_conv)

# Get test images
test_dataset = SyntheticImageDataset(n_samples=6, n_classes=10, img_size=32)
test_loader  = DataLoader(test_dataset, batch_size=6, shuffle=False)
test_imgs, test_labels = next(iter(test_loader))

# Generate Grad-CAM for each test image
fig, axes = plt.subplots(3, 6, figsize=(18, 9))

for i in range(6):
    single_img = test_imgs[i:i+1].to(device).requires_grad_(True)

    cam, pred_class, confidence = gradcam.generate(single_img)

    # Original image (unnormalized)
    img_np = test_imgs[i].permute(1, 2, 0).numpy()
    img_np = np.clip(img_np, 0, 1)

    overlay, heatmap = overlay_cam(img_np, cam, alpha=0.5)

    # Plot row 1: original
    axes[0, i].imshow(img_np)
    axes[0, i].set_title(f'Class {int(test_labels[i])}', fontsize=9)
    axes[0, i].axis('off')

    # Plot row 2: heatmap
    axes[1, i].imshow(cam, cmap='jet')
    axes[1, i].set_title(f'Pred:{pred_class} ({confidence:.1%})', fontsize=9)
    axes[1, i].axis('off')

    # Plot row 3: overlay
    axes[2, i].imshow(overlay)
    axes[2, i].set_title('Overlay', fontsize=9)
    axes[2, i].axis('off')

plt.suptitle('Grad-CAM: Row 1=Original | Row 2=Attention Heatmap | Row 3=Overlay\n'
             'Red/warm = regions that drove the prediction', fontsize=12)
plt.tight_layout()
plt.show()

print("Grad-CAM use cases:")
print("  • Debug wrong predictions (understand why model failed)")
print("  • Build trust with stakeholders (show what model focuses on)")
print("  • Medical AI (highlight suspicious regions in X-rays/MRI)")

---
## 8. Mini Project — Plant Disease Classifier <a id='8-mini-project'></a>

### What We're Building

A transfer-learned CNN that classifies plant leaf images into 4 disease categories. Real-world application: farmers photograph leaves with a phone → AI instantly identifies the disease → prescribes treatment.

**Dataset used in production:** PlantVillage (54,305 images, 38 classes) — available on Kaggle.

In [ ]:
# ── Synthetic plant disease dataset ──────────────────────────────────────────

PLANT_CLASSES = ['Healthy', 'Bacterial Blight', 'Rust Fungus', 'Mosaic Virus']
N_PLANT_CLASSES = len(PLANT_CLASSES)

class PlantDiseaseDataset(Dataset):
    """Synthetic plant disease dataset with class-specific visual patterns."""

    # Color signatures per disease (BGR → RGB converted below)
    CLASS_COLORS = [
        np.array([30, 150, 40]),    # Healthy: bright green
        np.array([80, 100, 30]),    # Bacterial Blight: dark green + brown
        np.array([160, 100, 40]),   # Rust Fungus: orange-brown
        np.array([120, 160, 50]),   # Mosaic Virus: mottled yellow-green
    ]

    def __init__(self, n_per_class=300, img_size=64, transform=None, split='train'):
        self.n = n_per_class * N_PLANT_CLASSES
        self.img_size = img_size
        self.transform = transform

        self.imgs = []
        self.labels = []

        rng = np.random.default_rng(42 if split == 'train' else 99)

        for cls_idx, base_color in enumerate(self.CLASS_COLORS):
            for _ in range(n_per_class):
                # Generate leaf-like image
                img = np.ones((img_size, img_size, 3), dtype=np.uint8) * 240

                # Leaf body
                noise = rng.integers(-30, 30, (img_size, img_size, 3))
                leaf = np.clip(base_color + noise, 0, 255).astype(np.uint8)
                img = leaf.copy()

                # Disease-specific patterns
                if cls_idx == 1:  # Bacterial Blight: dark spots
                    n_spots = rng.integers(3, 10)
                    for _ in range(n_spots):
                        cx = rng.integers(5, img_size-5)
                        cy = rng.integers(5, img_size-5)
                        r  = rng.integers(2, 8)
                        img[max(0,cy-r):cy+r, max(0,cx-r):cx+r] = [20, 20, 10]

                elif cls_idx == 2:  # Rust: orange patches
                    n_patches = rng.integers(4, 12)
                    for _ in range(n_patches):
                        cx = rng.integers(0, img_size)
                        cy = rng.integers(0, img_size)
                        r  = rng.integers(3, 10)
                        ys, xs = np.ogrid[:img_size, :img_size]
                        mask = (xs-cx)**2 + (ys-cy)**2 <= r**2
                        img[mask] = np.clip([200, 100, 30] + rng.integers(-20,20,3), 0, 255)

                elif cls_idx == 3:  # Mosaic: striped pattern
                    for row in range(0, img_size, rng.integers(3, 8)):
                        if rng.random() > 0.5:
                            img[row:row+2, :] = np.clip([200, 220, 50] + rng.integers(-20,20,3), 0, 255)

                self.imgs.append(img)
                self.labels.append(cls_idx)

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img = Image.fromarray(self.imgs[idx])
        if self.transform:
            img = self.transform(img)
        else:
            img = transforms.ToTensor()(img)
        return img, self.labels[idx]


plant_train_tfm = transforms.Compose([
    transforms.Resize(64),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

plant_val_tfm = transforms.Compose([
    transforms.Resize(64),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

plant_train = PlantDiseaseDataset(n_per_class=300, split='train', transform=plant_train_tfm)
plant_val   = PlantDiseaseDataset(n_per_class=80,  split='val',   transform=plant_val_tfm)

plant_train_loader = DataLoader(plant_train, batch_size=32, shuffle=True,  num_workers=0)
plant_val_loader   = DataLoader(plant_val,   batch_size=32, shuffle=False, num_workers=0)

print(f"Plant disease dataset:")
print(f"  Train: {len(plant_train)} images ({len(plant_train)//N_PLANT_CLASSES} per class)")
print(f"  Val:   {len(plant_val)} images")
print(f"  Classes: {PLANT_CLASSES}")

# Visualize sample leaves
fig, axes = plt.subplots(1, N_PLANT_CLASSES, figsize=(12, 4))
for cls_idx in range(N_PLANT_CLASSES):
    img = plant_train.imgs[cls_idx * 300]
    axes[cls_idx].imshow(img)
    axes[cls_idx].set_title(PLANT_CLASSES[cls_idx], fontsize=10)
    axes[cls_idx].axis('off')
plt.suptitle('Synthetic Plant Disease Classes', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Build & train plant disease classifier ────────────────────────────────────

# Lightweight model for 64×64 input
class PlantNet(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.2),   # 64→32

            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.2),   # 32→16

            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),                       # 16→8
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        return self.head(self.features(x))


plant_model = PlantNet(N_PLANT_CLASSES).to(device)
plant_opt   = optim.Adam(plant_model.parameters(), lr=5e-4, weight_decay=1e-4)
plant_sched = optim.lr_scheduler.ReduceLROnPlateau(plant_opt, patience=2, factor=0.5)

best_val_acc = 0
plant_history = {'train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}

print(f"{'Epoch':>6} {'Train Loss':>11} {'Train Acc%':>11} {'Val Loss':>10} {'Val Acc%':>9}")
print("─" * 56)

for epoch in range(1, 13):
    tr_loss, tr_acc = train_epoch(plant_model, plant_train_loader, plant_opt, criterion, device)
    va_loss, va_acc = eval_epoch(plant_model, plant_val_loader, criterion, device)
    plant_sched.step(va_loss)

    plant_history['train_acc'].append(tr_acc)
    plant_history['val_acc'].append(va_acc)
    plant_history['train_loss'].append(tr_loss)
    plant_history['val_loss'].append(va_loss)

    flag = ' ← best' if va_acc > best_val_acc else ''
    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save(plant_model.state_dict(), '/tmp/plant_model_best.pt')

    print(f"{epoch:>6} {tr_loss:>11.4f} {tr_acc*100:>11.2f} {va_loss:>10.4f} {va_acc*100:>9.2f}{flag}")

print(f"\nBest validation accuracy: {best_val_acc*100:.2f}%")

In [ ]:
# ── Confusion matrix & per-class report ──────────────────────────────────────

# Load best model
plant_model.load_state_dict(torch.load('/tmp/plant_model_best.pt', map_location=device))
plant_model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in plant_val_loader:
        imgs = imgs.to(device)
        logits = plant_model(imgs)
        preds = logits.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Confusion matrix
cm_matrix = confusion_matrix(all_labels, all_preds)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=PLANT_CLASSES, yticklabels=PLANT_CLASSES, ax=axes[0])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].set_title('Confusion Matrix')
plt.setp(axes[0].get_xticklabels(), rotation=30, ha='right')

# Training curves
epochs = range(1, 13)
axes[1].plot(epochs, [a*100 for a in plant_history['train_acc']], 'b-o', ms=4, label='Train')
axes[1].plot(epochs, [a*100 for a in plant_history['val_acc']],   'r-o', ms=4, label='Val')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Plant Disease Classifier Training')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Per-class report:")
print(classification_report(all_labels, all_preds, target_names=PLANT_CLASSES))

print("\nReal-world business value:")
print("  • Farmer photographs leaf → instant disease diagnosis")
print("  • Works offline on mobile phone (convert to TorchScript/ONNX)")
print("  • Scales to 38 PlantVillage classes with same architecture")

---
## 9. Common Pitfalls <a id='9-pitfalls'></a>

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║             TORCHVISION — COMMON PITFALLS                        ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  1. WRONG: Same transform for train and validation               ║
║     dataset = ImageFolder('data', transform=train_transform)     ║
║     # Augmentation on val → artificially inflated variance       ║
║  RIGHT: train and val MUST have separate transforms              ║
║     train_ds = ImageFolder('data/train', transform=train_tfm)    ║
║     val_ds   = ImageFolder('data/val',   transform=val_tfm)      ║
║                                                                  ║
║  2. WRONG: Forgetting to use .eval() for validation              ║
║     model.train()   ← left in training mode                      ║
║     # BatchNorm + Dropout behave differently in train vs eval!   ║
║  RIGHT:                                                          ║
║     model.eval()                                                 ║
║     with torch.no_grad():   ← also saves GPU memory              ║
║         ...                                                      ║
║                                                                  ║
║  3. WRONG: Fine-tuning at the same LR as head training           ║
║     optimizer = optim.Adam(model.parameters(), lr=1e-3)          ║
║     # Destroys pre-trained weights of backbone!                  ║
║  RIGHT: Phase 2 LR should be 10-100x smaller                    ║
║     optimizer = optim.Adam(model.parameters(), lr=1e-4)          ║
║                                                                  ║
║  4. WRONG: Not normalizing with model's expected stats           ║
║     transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])          ║
║     # ResNet/EfficientNet pre-trained on ImageNet!               ║
║  RIGHT: use ImageNet stats                                       ║
║     transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]) ║
║                                                                  ║
║  5. WRONG: Passing gradient to val DataLoader images             ║
║     logits = model(imgs)   ← inside eval loop, no no_grad ctx   ║
║     # Builds computation graph unnecessarily → memory waste      ║
║  RIGHT: always wrap val loop in torch.no_grad()                  ║
║                                                                  ║
║  6. WRONG: Using model output directly as probabilities          ║
║     probs = model(imgs)   ← these are RAW LOGITS, not probs!     ║
║  RIGHT: apply softmax first                                      ║
║     probs = F.softmax(model(imgs), dim=1)                        ║
║     # Or use nn.CrossEntropyLoss which includes log_softmax      ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
""")

---
## 10. Interview Q&A <a id='10-interview'></a>

In [ ]:
qa = [
    ("What is transfer learning and why does it work for images?",
     "Transfer learning reuses a model pre-trained on a large dataset (e.g., ImageNet) for a "
     "new, smaller dataset. It works because CNNs learn hierarchical features: early layers "
     "detect universal low-level features (edges, corners, colors) regardless of domain, "
     "while later layers learn domain-specific features (fur texture, wheel shapes). "
     "These early features transfer well to new domains. Benefits: requires far less data, "
     "trains much faster, and achieves better accuracy than training from scratch."),

    ("What does BatchNorm do and why does it help?",
     "BatchNorm normalizes activations within each mini-batch to have mean≈0, std≈1, "
     "then applies learnable scale (gamma) and shift (beta). Benefits: (1) Reduces "
     "internal covariate shift — stabilizes training; (2) Acts as regularizer — reduces "
     "need for dropout; (3) Allows higher learning rates; (4) Reduces sensitivity to "
     "weight initialization. Critical: behaves differently in train (uses batch stats) "
     "vs eval (uses running mean/var computed during training). Always call model.eval()."),

    ("Explain the difference between Global Average Pooling and Flatten.",
     "Both convert feature maps to a vector for the classifier head. "
     "Flatten: (C, H, W) → (C×H×W) — concatenates all values. The input spatial size "
     "must be fixed (model tied to image size). "
     "GAP: (C, H, W) → (C) — averages each channel's spatial values. "
     "Benefits of GAP: (1) any input size works; (2) far fewer parameters in FC layer; "
     "(3) acts as structural regularizer; (4) less prone to overfitting. "
     "GAP is standard in modern architectures (ResNet, EfficientNet, MobileNet)."),

    ("Why use data augmentation and what transforms are most important?",
     "Augmentation artificially expands the training set, reducing overfitting and improving "
     "generalization. Most important transforms: (1) RandomHorizontalFlip — usually safe for "
     "most domains (NOT for medical text recognition); (2) RandomCrop — removes position bias; "
     "(3) ColorJitter — makes model robust to lighting changes; (4) RandomRotation — for "
     "scenes where orientation varies. Avoid: flips/rotations that produce impossible images "
     "(e.g., upside-down dogs, mirrored text)."),

    ("What is Grad-CAM and when would you use it?",
     "Gradient-weighted Class Activation Mapping visualizes which spatial regions of an input "
     "image most influenced a CNN's prediction. Computed by: (1) forward pass through last "
     "conv layer; (2) backward pass from target class score; (3) global-average-pool gradients "
     "→ class weights; (4) weighted sum of activation maps → ReLU → heatmap. "
     "Used for: debugging wrong predictions, building stakeholder trust, "
     "regulatory compliance in medical AI, and detecting dataset bias."),

    ("How does learning rate affect transfer learning fine-tuning?",
     "Phase 1 (head only): use standard LR (1e-3 to 1e-4). The pre-trained backbone is frozen "
     "and won't be affected. Phase 2 (fine-tune backbone): MUST use very small LR (1e-5 to 1e-4). "
     "Too high LR destroys carefully pre-trained weights via catastrophic forgetting. "
     "Best practice: use discriminative learning rates — different LR per layer group. "
     "Later layers: standard LR. Earlier layers: 10-100× smaller LR."),
]

print("=" * 70)
print("  INTERVIEW Q&A — TORCHVISION")
print("=" * 70)
for i, (q, a) in enumerate(qa, 1):
    print(f"\nQ{i}: {q}")
    print(f"\nA{i}: {a}")
    print("\n" + "─" * 70)

---
## 11. Resources <a id='11-resources'></a>

### Official Documentation
- **Torchvision**: https://pytorch.org/vision/stable/
- **Pre-trained models**: https://pytorch.org/vision/stable/models.html
- **Transforms v2 (new API)**: https://pytorch.org/vision/stable/transforms.html

### Research Papers
- **ResNet** (He et al., 2015): https://arxiv.org/abs/1512.03385
- **EfficientNet** (Tan & Le, 2019): https://arxiv.org/abs/1905.11946
- **Grad-CAM**: https://arxiv.org/abs/1610.02391
- **Vision Transformer (ViT)**: https://arxiv.org/abs/2010.11929

### Video Tutorials
- **PyTorch for Computer Vision**: https://youtu.be/pDdP0TFzsoQ
- **Transfer learning tutorial** (official): https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html
- **CNN architectures explained**: https://youtu.be/ACmydtFDTGs

### Datasets
- **PlantVillage** (plant diseases): https://www.kaggle.com/datasets/emmarex/plantdisease
- **ImageNet**: https://www.image-net.org/
- **CIFAR-10/100**: Automatically downloaded via `datasets.CIFAR10(download=True)`
- **Torchvision built-in datasets**: https://pytorch.org/vision/stable/datasets.html

---
## 12. Summary & What's Next <a id='12-summary'></a>

### What You Learned

| Concept | Key Takeaway |
|---|---|
| Transforms | Augmentation for train, normalize-only for val; `ToTensor()` → (C,H,W), [0,1] |
| Datasets | `ImageFolder` for custom data; `DataLoader` for batching + shuffling |
| Pre-trained models | Start with ResNet-18/50; use `weights=...IMAGENET1K_V1` |
| Transfer learning | Freeze backbone → train head → unfreeze → fine-tune at 10× lower LR |
| CNN from scratch | Use BN + Dropout + GAP; only when domain differs drastically |
| Grad-CAM | Gradient-weighted heatmap showing what the model saw |
| Training loop | train_epoch + eval_epoch; model.train() / model.eval() |

### Golden Rules

1. **Always use transfer learning** unless you have a very good reason not to
2. **Never augment your validation set** — only normalize
3. **Always call `model.eval()` + `torch.no_grad()`** during validation
4. **Fine-tune at 10×-100× lower LR** than initial training

### What's Next

| Notebook | Topic |
|---|---|
| `YOLO_Ultralytics` | Real-time object detection — finding WHERE objects are |
| `Detectron2` | Instance segmentation — pixel-perfect object masks |

**Classification answers "what?" — Detection answers "what AND where?" — that's next!**